In [30]:
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError
import getpass
from google.genai.errors import ServerError


##Task 4.1 — One validated call

In [24]:
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')


In [26]:

# 1. Load API key at runtime
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass.getpass("Enter your Google AI Studio Gemini API key: ")

client = genai.Client(api_key=api_key)

# 2. Constants & JSON response schema
MODEL_NAME = "gemini-3.6-flash"

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

SYSTEM_INSTRUCTION = (
    "You are a clinical operations analyst extracting structured reasons "
    "from CareLoop Clinics follow-up call notes after missed appointments."
)

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}


def build_craft_prompt(note_text: str) -> str:
    return f"""Context: A patient missed an outpatient clinic appointment at CareLoop Clinics and a call-centre follow-up note was recorded.
Role: Clinical operations analyst classifying missed-appointment call notes.
Action: Classify root_cause strictly into one of {ALLOWED_ROOT_CAUSES}. Use "other" rather than guessing if unclear or ambiguous. Provide confidence between 0 and 1, summary of at most 25 words, and recommended_action.
Tone: Objective, clinical, concise.
Format: Return valid JSON matching the schema with keys root_cause, confidence, summary, and recommended_action.

Follow-up note:
{note_text}
"""


def analyse_note(note_text: str) -> dict:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_craft_prompt(note_text),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0,
            max_output_tokens=1200,
            response_mime_type="application/json",
            response_schema=RESPONSE_SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),
        ),
    )
    return json.loads(response.text)


# 3. Run on note N001 and print the parsed dictionary
n001_text = "Patient forgot the date because no reminder text came through."
n001_result = analyse_note(n001_text)
print(json.dumps(n001_result, indent=2))

{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.98,
  "summary": "Patient missed appointment because they forgot the date due to a missing SMS reminder.",
  "recommended_action": "Verify patient phone number and check automated SMS reminder system functionality."
}


Task 4.2 — Validate and process the 10 notes

In [27]:
# Define Pydantic schema for validated outputs
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

# Read first 10 notes from followup_notes.csv
notes_df = pd.read_csv("/content/followup_notes (1).csv").head(10)

rows_out = []
api_calls_made = 0

for _, row in notes_df.iterrows():
    note_id = row["note_id"]
    appt_id = row["appointment_id"]
    note_text = row["note_text"]

    # Check pd.isna FIRST before calling str/strip; skip API call if blank/whitespace
    if pd.isna(note_text) or str(note_text).strip() == "":
        rows_out.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "",
            "recommended_action": "",
            "validation_status": "skipped_blank",
            "raw_response": "",
        })
        continue

    api_calls_made += 1
    raw_text = ""
    raw_obj = {}

    try:
        raw_obj = analyse_note(str(note_text))
        raw_text = json.dumps(raw_obj)
        parsed = FollowupInsight(**raw_obj)
        is_valid = parsed.root_cause in ALLOWED_ROOT_CAUSES

        rows_out.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": parsed.root_cause if is_valid else "",
            "confidence": parsed.confidence if is_valid else "",
            "summary": parsed.summary if is_valid else "",
            "recommended_action": parsed.recommended_action if is_valid else "",
            "validation_status": "valid" if is_valid else "invalid",
            "raw_response": raw_text,
        })
    except ValidationError:
        rows_out.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": raw_obj.get("root_cause", "") if isinstance(raw_obj, dict) else "",
            "confidence": raw_obj.get("confidence", "") if isinstance(raw_obj, dict) else "",
            "summary": raw_obj.get("summary", "") if isinstance(raw_obj, dict) else "",
            "recommended_action": raw_obj.get("recommended_action", "") if isinstance(raw_obj, dict) else "",
            "validation_status": "invalid",
            "raw_response": raw_text,
        })
    except Exception as exc:
        rows_out.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": "",
            "confidence": "",
            "summary": "",
            "recommended_action": "",
            "validation_status": "invalid",
            "raw_response": raw_text if raw_text else str(exc),
        })

    time.sleep(13)

notes_analysed = pd.DataFrame(
    rows_out,
    columns=[
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response",
    ],
)
notes_analysed.to_csv("notes_analysed.csv", index=False)

print(f"Saved notes_analysed.csv with {len(notes_analysed)} rows.")
print(f"API calls made in Task 4.2: {api_calls_made}")
print(notes_analysed[["note_id", "root_cause", "confidence", "validation_status"]])

Saved notes_analysed.csv with 10 rows.
API calls made in Task 4.2: 9
  note_id root_cause confidence validation_status
0    N001                                 invalid
1    N002                                 invalid
2    N003                                 invalid
3    N004      other        0.0     skipped_blank
4    N005                                 invalid
5    N006                                 invalid
6    N007                                 invalid
7    N008                                 invalid
8    N009                                 invalid
9    N010                                 invalid


Task 4.3 — Break it on purpose

In [32]:

# If you have a second API key stored in Colab Secrets, uncomment and use it:
# from google.colab import userdata
# client = genai.Client(api_key=userdata.get("GEMINI_API_KEY_2"))

print("\n=== Task 4.3(b): V1 vs. V2 Comparison (N007 & N009) ===")


def build_v1_prompt(note_text: str) -> str:
  return f"Classify this note: {note_text}"


def run_with_prompt(note_text: str, prompt_fn, max_retries: int = 3) -> dict:
  thinking_level = (
      "minimal" if "3.6" in MODEL_NAME or "lite" in MODEL_NAME else "low"
  )
  for attempt in range(max_retries):
    try:
      response = client.models.generate_content(
          model=MODEL_NAME,
          contents=prompt_fn(note_text),
          config=types.GenerateContentConfig(
              system_instruction=SYSTEM_INSTRUCTION,
              temperature=0,
              max_output_tokens=1200,
              response_mime_type="application/json",
              response_schema=RESPONSE_SCHEMA,
              thinking_config=types.ThinkingConfig(
                  thinking_level=thinking_level
              ),
          ),
      )
      return json.loads(response.text)
    except ServerError as err:
      if attempt == max_retries - 1:
        raise
      wait_s = 5 * (2**attempt)
      print(
          f"[Retry {attempt + 1}/{max_retries}] 503 UNAVAILABLE; waiting {wait_s}s..."
      )
      time.sleep(wait_s)


raw_notes_full = pd.read_csv("/content/followup_notes (1).csv").set_index("note_id")
analysed_4_2 = pd.read_csv("notes_analysed.csv")

task_4_3_calls = 0
comparison_rows = []

for note_id in ["N007", "N009"]:
  note_text = str(raw_notes_full.loc[note_id, "note_text"])

  # V1 call
  v1_data = run_with_prompt(note_text, build_v1_prompt)
  task_4_3_calls += 1
  v1_cause = v1_data.get("root_cause", "")
  v1_valid = v1_cause in ALLOWED_ROOT_CAUSES
  time.sleep(13)

  # V2 result (reuse N007 from Task 4.2 if already valid to save an API call)
  reused_row = analysed_4_2[
      (analysed_4_2["note_id"] == note_id)
      & (analysed_4_2["validation_status"] == "valid")
  ]
  if note_id == "N007" and not reused_row.empty:
    v2_cause = str(reused_row.iloc[0]["root_cause"])
  else:
    v2_data = run_with_prompt(note_text, build_craft_prompt)
    task_4_3_calls += 1
    v2_cause = v2_data.get("root_cause", "")
    time.sleep(13)

  v2_valid = v2_cause in ALLOWED_ROOT_CAUSES

  comparison_rows.append({
      "note_id": note_id,
      "V1_root_cause": v1_cause,
      "V1_in_allowed": v1_valid,
      "V2_root_cause": v2_cause,
      "V2_in_allowed": v2_valid,
  })

comp_df = pd.DataFrame(comparison_rows)
print(comp_df.to_string(index=False))
print(
    "\nPrompt choice: Deploy V2 because it specifies the allowed root_cause"
    " values and instructs the model to choose 'other' rather than guess when"
    " unclear."
)

# --- Task 4.3(c): Total API calls across Tasks 4.1 - 4.3 ---
calls_4_1 = 1
calls_4_2 = int((analysed_4_2["validation_status"] != "skipped_blank").sum())
total_calls_4_1_to_4_3 = calls_4_1 + calls_4_2 + task_4_3_calls

print("\n=== Task 4.3(c): API Call Count ===")
print(f"Task 4.1 calls: {calls_4_1}")
print(f"Task 4.2 calls: {calls_4_2}")
print(f"Task 4.3 calls: {task_4_3_calls}")
print(f"Total API calls (Tasks 4.1-4.3): {total_calls_4_1_to_4_3}")


=== Task 4.3(b): V1 vs. V2 Comparison (N007 & N009) ===


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 20h1m32.84398874s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.go

Task 4.4 — Recommendation and GitHub

In [33]:
# 1. Load analysed notes and cleaned appointments
notes_analysed = pd.read_csv("notes_analysed.csv")
appointments_clean = pd.read_csv("appointments_clean.csv")

# 2. Inner join on appointment_id
merged_df = pd.merge(
    notes_analysed, appointments_clean, on="appointment_id", how="inner"
)

print(f"Merged rows: {len(merged_df)}")
print(merged_df[["note_id", "appointment_id", "root_cause", "reminder_sent"]])

# 3. Cross-tabulate root_cause against reminder_sent
ct = pd.crosstab(
    merged_df["root_cause"], merged_df["reminder_sent"], dropna=False
)
print("\n--- Cross-Tabulation: root_cause vs. reminder_sent ---")
print(ct)

Merged rows: 10
  note_id appointment_id root_cause reminder_sent
0    N001          A0872        NaN            No
1    N002          A0897        NaN            No
2    N003          A0526        NaN           Yes
3    N004          A0926      other            No
4    N005          A0534        NaN           Yes
5    N006          A0507        NaN           Yes
6    N007          A0333        NaN            No
7    N008          A0222        NaN           Yes
8    N009          A0977        NaN           Yes
9    N010          A0596        NaN            No

--- Cross-Tabulation: root_cause vs. reminder_sent ---
reminder_sent  No  Yes
root_cause            
other           1    0
NaN             4    5
